# Paytm Statement Parser — with PostgreSQL Storage

This notebook:
1. Parses `statement.pdf` into a clean structured DataFrame
2. Extracts merchant name from raw details
3. Converts Amount to a numeric float with a Debit/Credit type column
4. Attaches the correct year to each transaction date using the statement's own header
5. Categorizes each transaction using a generalized keyword dictionary (`categories.json`)
6. Stores data in PostgreSQL — per account holder, deduplicating on UPI Ref No (internally)
7. Exports only model-safe columns to `transactions.csv`

**Columns removed (PII / useless for modeling):** `UPI_ID`, `UPI_Ref`, `Account`, `Details`

In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────
import os

PDF_PATH   = 'statement.pdf'
DB_HOST    = os.getenv('DB_HOST', 'localhost')
DB_PORT    = int(os.getenv('DB_PORT', '5432'))
DB_NAME    = os.getenv('DB_NAME', 'personal_finance')
DB_USER    = os.getenv('DB_USER', 'postgres')
DB_PASS    = os.environ['DB_PASSWORD']
CSV_OUTPUT = 'transactions.csv'
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import pdfplumber
import re
import json
import pandas as pd
import psycopg2
from datetime import datetime

# ── 1. LOAD CATEGORY MAPPING FROM EXTERNAL FILE ──────────────────────────────
with open('categories.json', 'r') as f:
    CATEGORY_MAPPING = json.load(f)

print(f"Loaded {len(CATEGORY_MAPPING)} categories from categories.json")

In [ ]:
# ── 2. PDF PARSING HELPERS ────────────────────────────────────────────────────
date_pattern = re.compile(r'^\d+\s+[A-Za-z]{3}$')
time_pattern = re.compile(r'^\d+:\d+\s+(?:AM|PM)$')
col_boundaries = [85, 390, 485]  # [Date, Details, Account, Amount]

SKIP_KEYWORDS = [
    'passbook payments history', 'all payments done', 'date &',
    'transaction details', 'notes & tags', 'your account', 'amount',
    'page ', 'for any queries', 'contact us', 'paytm statement for',
    'total money', 'payments made', 'self transfer',
    'payments that you might', 'paytm payments bank wallet',
    'accounts payment made', 'state bank of india - 30 rs',
    '(3 payments)', '(2 payments)'
]

def is_noise(cols):
    joined = ' '.join(cols).lower()
    return any(kw in joined for kw in SKIP_KEYWORDS)

def parse_statement(pdf_path, col_boundaries):
    """Modular PDF parser — works for any bank by adjusting col_boundaries."""
    transactions, current_tx = [], None
    meta = {'phone': None, 'name': None, 'date_range': None}

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            words = page.extract_words()
            lines = []
            for w in words:
                placed = False
                for line in lines:
                    if abs(line[0]['top'] - w['top']) < 3:
                        line.append(w)
                        placed = True
                        break
                if not placed:
                    lines.append([w])

            lines.sort(key=lambda l: l[0]['top'])

            for line in lines:
                line.sort(key=lambda w: w['x0'])
                raw_text = ' '.join(w['text'] for w in line)

                phone_match = re.search(r'(\d{10})', raw_text)
                if phone_match and not meta['phone']:
                    meta['phone'] = phone_match.group(1)

                range_match = re.search(r"(\d+\s+[A-Z]+'\d+)\s*-\s*(\d+\s+[A-Z]+'\d+)", raw_text)
                if range_match and not meta['date_range']:
                    meta['date_range'] = (range_match.group(1), range_match.group(2))

                if 'Paytm User' in raw_text:
                    meta['name'] = 'Paytm User'

                cols = ['', '', '', '']
                for w in line:
                    x0 = w['x0']
                    col_idx = 0
                    while col_idx < len(col_boundaries) and x0 >= col_boundaries[col_idx]:
                        col_idx += 1
                    cols[col_idx] = (cols[col_idx] + ' ' + w['text']).strip()

                if is_noise(cols):
                    continue

                val0 = cols[0]
                if date_pattern.match(val0):
                    if current_tx:
                        transactions.append(current_tx)
                    current_tx = {
                        'Date': val0, 'Time': '',
                        'Details': cols[1], 'Account': cols[2], 'Amount_raw': cols[3]
                    }
                elif current_tx:
                    if time_pattern.match(val0):
                        current_tx['Time'] = val0
                    if cols[1]:
                        current_tx['Details'] = (current_tx['Details'] + ' ' + cols[1]).strip()
                    if cols[2]:
                        current_tx['Account'] = (current_tx['Account'] + ' ' + cols[2]).strip()
                    if cols[3]:
                        current_tx['Amount_raw'] = (current_tx['Amount_raw'] + ' ' + cols[3]).strip()

        if current_tx:
            transactions.append(current_tx)

    return transactions, meta

In [ ]:
# ── 3. ENRICHMENT HELPERS ─────────────────────────────────────────────────────

def extract_merchant(details):
    """Extract clean merchant/person name from raw details text."""
    match = re.search(r'(?:Paid to|Received from|Cashback Received from)\s+([^:\n#]+)', details, re.IGNORECASE)
    if match:
        org = match.group(1).strip()
        org = re.split(r'\b(?:Note|UPI|Tag|Ref|using|on|via)\b', org, flags=re.IGNORECASE)[0].strip()
        org = re.sub(r'\s+(Limited|Private|Pvt)$', '', org, flags=re.IGNORECASE)
        return org.strip()
    return 'Unknown'

def _extract_upi_ref_internal(details):
    """Internal only — used for DB deduplication. Never exposed in the model DataFrame."""
    match = re.search(r'UPI Ref No:\s*(\d+)', details, re.IGNORECASE)
    return match.group(1).strip() if match else None

def parse_amount(amount_raw):
    """Convert raw amount string like '+ Rs.103.96' to numeric float and Debit/Credit type."""
    s = amount_raw.strip() if amount_raw else ''
    is_credit = s.startswith('+')
    m = re.search(r'[\d,]+\.?\d*', s)
    if not m:
        return (None, None)
    cleaned = m.group(0).replace(',', '')
    try:
        val = float(cleaned)
        return (val, 'Credit') if is_credit else (-val, 'Debit')
    except ValueError:
        return (None, None)

def infer_year(date_str, date_range):
    """Attach the correct year to a date like '23 Aug' using the statement's date range header."""
    if not date_range:
        return date_str + f" {datetime.now().year}"
    try:
        start = datetime.strptime(date_range[0].replace("'", " 20"), "%d %b  20%y")
        end   = datetime.strptime(date_range[1].replace("'", " 20"), "%d %b  20%y")
        tx_date = datetime.strptime(date_str, "%d %b")
        for year in [end.year, start.year]:
            candidate = tx_date.replace(year=year)
            if start <= candidate <= end:
                return date_str + f" {year}"
        return date_str + f" {end.year}"
    except Exception:
        return date_str + f" {datetime.now().year}"

def categorize(merchant, details):
    """Two-stage generalized categorization using the CATEGORY_MAPPING dictionary."""
    merchant_l, details_l = merchant.lower(), details.lower()
    for category, keywords in CATEGORY_MAPPING.items():
        if any(kw in merchant_l for kw in keywords):
            return category
    for category, keywords in CATEGORY_MAPPING.items():
        if any(kw in details_l for kw in keywords):
            return category
    return 'Others/Uncategorized'

In [ ]:
# ── 4. PARSE PDF AND BUILD DATAFRAME ─────────────────────────────────────────
raw_txns, meta = parse_statement(PDF_PATH, col_boundaries)

print(f"Account holder : {meta['name']}")
print(f"Phone number   : {meta['phone']}")
print(f"Statement range: {meta['date_range']}")
print(f"Transactions   : {len(raw_txns)}")

records = []
for tx in raw_txns:
    merchant        = extract_merchant(tx['Details'])
    _upi_ref        = _extract_upi_ref_internal(tx['Details'])  # internal only, not in model DataFrame
    amount, tx_type = parse_amount(tx['Amount_raw'])
    date_full       = infer_year(tx['Date'], meta['date_range'])
    category        = categorize(merchant, tx['Details'])

    records.append({
        # ── Model-safe columns ──────────────────────────
        'Date'    : date_full,
        'Time'    : tx['Time'],
        'Merchant': merchant,
        'Amount'  : amount,
        'Type'    : tx_type,
        'Category': category,
        # ── Internal DB key (not in model DataFrame) ───
        '_upi_ref': _upi_ref
    })

df = pd.DataFrame(records)

# Model-facing view — drop internal DB key
df_model = df.drop(columns=['_upi_ref'])
df_model

In [ ]:
# ── 5. STORE IN POSTGRESQL ────────────────────────────────────────────────────
conn = psycopg2.connect(
    host=DB_HOST, port=DB_PORT,
    dbname=DB_NAME, user=DB_USER, password=DB_PASS
)
cur = conn.cursor()

phone = meta['phone']
name  = meta['name'] or 'Unknown'

# Upsert account holder — create if new user, skip if existing
cur.execute("""
    INSERT INTO account_holders (phone_number, name)
    VALUES (%s, %s)
    ON CONFLICT (phone_number) DO NOTHING
    RETURNING id;
""", (phone, name))

row = cur.fetchone()
if row:
    holder_id = row[0]
    print(f"New account holder created  → id={holder_id}, phone={phone}")
else:
    cur.execute("SELECT id FROM account_holders WHERE phone_number = %s", (phone,))
    holder_id = cur.fetchone()[0]
    print(f"Existing account holder found → id={holder_id}, phone={phone}")

# Insert transactions — skip duplicates using UPI Ref No as unique key
inserted, skipped = 0, 0
for _, row in df.iterrows():
    try:
        cur.execute("""
            INSERT INTO transactions
                (holder_id, date, time, merchant, upi_ref, amount, type, category)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (upi_ref) DO NOTHING;
        """, (
            holder_id,
            row['Date'], row['Time'],
            row['Merchant'], row['_upi_ref'],
            row['Amount'], row['Type'], row['Category']
        ))
        if cur.rowcount > 0:
            inserted += 1
        else:
            skipped += 1
    except Exception as e:
        print(f"Error inserting row: {e}")
        conn.rollback()

conn.commit()
cur.close()
conn.close()

print(f"\n✅ Done  →  {inserted} inserted,  {skipped} skipped (already exist)")

In [ ]:
# ── 6. EXPORT MODEL-SAFE COLUMNS TO CSV ──────────────────────────────────────
df_model.to_csv(CSV_OUTPUT, index=False)
print(f"Saved {len(df_model)} rows to {CSV_OUTPUT}")
print(f"Columns: {list(df_model.columns)}")
df_model